# 02 — Exploratory Data Analysis (EDA)
**ChurnOps Phase 3** — Deep Data Discovery & Behavioral Characterization  
*All figures and statistics are computed live on the actual IBM Telco Customer Churn dataset.*

---

### Objectives
1. **Univariate Analysis:** Understand distribution, spread, and skewness of numeric features, and cardinality of categorical features.
2. **Target Distribution & Imbalance:** Measure class imbalance (2.77:1) and explain why accuracy is misleading in customer retention.
3. **Bivariate Analysis:** Uncover the primary operational and contractual drivers of churn.
4. **Association Metrics:** Pearson/Spearman for numerics; **Cramér's V** for categoricals (never Pearson on nominal data).
5. **Outlier Detection & Classification:** Use IQR and domain logic to classify extreme values as valid observations rather than auto-deleting them.
6. **Data Anomaly Verification:** Deep-dive into the 11 blank `TotalCharges` records flagged in Phase 2.
7. **Synthesis:** Produce actionable directives for Phase 4 (Data Cleaning) and Phase 5 (Feature Engineering).

In [1]:
import sys, os
from pathlib import Path

# Add project root to sys.path so src imports work seamlessly
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, chi2_contingency

from src.utils.config import get_path, PARAMS
from src.data.eda import load_and_prep_for_eda, calculate_cramers_v

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("Project Root:", PROJECT_ROOT)
print("Configuration Target Column:", PARAMS['dataset']['target_column'])

Project Root: C:\Users\Abhishek\OneDrive\Pictures\Desktop\ChurnOps
Configuration Target Column: Churn


In [2]:
# Load raw data and analysis copy (with TotalCharges converted to numeric for statistical analysis)
df_raw, df = load_and_prep_for_eda()
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
df.head(5)

Dataset Shape: 7,043 rows x 23 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TotalCharges_num,Churn_binary
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,29.85,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,One year,No,Mailed check,56.95,1889.5,No,1889.50,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,108.15,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,1840.75,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,151.65,1


## 1. Target Distribution & Class Imbalance Analysis

In churn prediction, false negatives (failing to identify a customer about to cancel) result in immediate, permanent revenue loss.
Here we evaluate the class balance and quantify why standard accuracy fails.

In [3]:
target_counts = df['Churn'].value_counts()
target_pcts = df['Churn'].value_counts(normalize=True) * 100
imbalance_ratio = target_counts['No'] / target_counts['Yes']

print(f"Retained (No): {target_counts['No']:,} ({target_pcts['No']:.2f}%)")
print(f"Churned (Yes) : {target_counts['Yes']:,} ({target_pcts['Yes']:.2f}%)")
print(f"Imbalance Ratio: {imbalance_ratio:.3f} : 1")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#4C72B0', '#DD8452']

sns.barplot(x=target_counts.index, y=target_counts.values, palette=colors, ax=ax1, hue=target_counts.index, legend=False)
ax1.set_title("Customer Retention Count", weight='bold')
ax1.set_ylabel("Number of Customers")
for p in ax1.patches:
    h = p.get_height()
    pct = (h / len(df)) * 100
    ax1.annotate(f"{int(h):,}\n({pct:.1f}%)", (p.get_x() + p.get_width()/2., h/2),
                 ha='center', va='center', color='white', weight='bold')

ax2.pie(target_counts.values, labels=[f"No (Retained)\n{target_counts['No']:,}", f"Yes (Churned)\n{target_counts['Yes']:,}"],
        autopct='%1.1f%%', colors=colors, startangle=90, explode=(0, 0.08),
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax2.set_title(f"Imbalance Ratio {imbalance_ratio:.2f}:1", weight='bold')
plt.tight_layout()
plt.show()

Retained (No): 5,174 (73.46%)
Churned (Yes) : 1,869 (26.54%)
Imbalance Ratio: 2.768 : 1


C:\Users\Abhishek\AppData\Local\Temp\ipykernel_36256\1456137074.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Why Naive Accuracy is Misleading
If a naive classifier simply predicts `"No"` for all customers:
- **Accuracy achieved:** $5,174 / 7,043 = 73.46\%$
- **Recall on Churners:** $0 / 1,869 = 0.0\%$
- **Business impact:** 100% of at-risk customers walk out the door unnoticed.

**Methodological mandate for downstream phases:**
1. **Stratification:** Stratified train/validation/test splits (Phase 7) to maintain the 26.54% minority ratio.
2. **Cost-sensitive learning:** Apply `class_weight='balanced'` on Logistic Regression & Random Forest, and `scale_pos_weight=2.77` on XGBoost (Phase 9).
3. **Evaluation Metrics:** Prioritize **Recall**, **F1-Score**, and **Precision-Recall AUC (PR-AUC)** over raw ROC-AUC or Accuracy (Phase 10 & 14).

## 2. Numeric Features: Univariate Distribution & Skewness

We inspect `tenure`, `MonthlyCharges`, and `TotalCharges` (parsed to numeric, omitting 11 blank records for distribution assessment).

In [4]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges_num']

summary_stats = []
for c in num_cols:
    s = df[c].dropna()
    q25, q75 = s.quantile(0.25), s.quantile(0.75)
    iqr = q75 - q25
    summary_stats.append({
        'Feature': c,
        'Mean': round(s.mean(), 2),
        'Std': round(s.std(), 2),
        'Median': round(s.median(), 2),
        'Min': round(s.min(), 2),
        'Max': round(s.max(), 2),
        'IQR': round(iqr, 2),
        'Skewness': round(skew(s), 3)
    })

pd.DataFrame(summary_stats)

,Feature,Mean,Std,Median,Min,Max,IQR,Skewness
0,tenure,32.37,24.56,29.00,0.00,72.00,46.00,0.239
1,MonthlyCharges,64.76,30.09,70.35,18.25,118.75,54.35,-0.220
2,TotalCharges_num,2283.30,2266.77,1397.48,18.80,8684.80,3393.29,0.961


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for idx, c in enumerate(num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=axes[idx], color='#2b5c8f', bins=30)
    axes[idx].set_title(f"{c} (Skew: {skew(df[c].dropna()):.2f})", weight='bold')
    axes[idx].set_xlabel(c)
plt.tight_layout()
plt.show()

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_36256\4260970010.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Categorical Features & Cardinality Analysis

In [6]:
cat_cols = [c for c in df.columns if c not in num_cols + ['customerID', 'Churn', 'Churn_binary', 'TotalCharges']]

cardinality_data = []
for c in cat_cols:
    cardinality_data.append({
        'Feature': c,
        'Cardinality': df[c].nunique(),
        'Unique Values': str(list(df[c].unique())),
        'Mode': df[c].mode()[0],
        'Mode %': round((df[c].value_counts().iloc[0] / len(df)) * 100, 1)
    })

pd.DataFrame(cardinality_data)

,Feature,Cardinality,Unique Values,Mode,Mode %
0,gender,2,"['Female', 'Male']",Male,50.5
1,SeniorCitizen,2,"[0, 1]",0,83.8
2,Partner,2,"['Yes', 'No']",No,51.7
3,Dependents,2,"['No', 'Yes']",No,70.0
4,PhoneService,2,"['No', 'Yes']",Yes,90.3
5,MultipleLines,3,"['No phone service', 'No', 'Yes']",No,48.1
6,InternetService,3,"['DSL', 'Fiber optic', 'No']",Fiber optic,44.0
7,OnlineSecurity,3,"['No', 'Yes', 'No internet service']",No,49.7
8,OnlineBackup,3,"['Yes', 'No', 'No internet service']",No,43.8
9,DeviceProtection,3,"['No', 'Yes', 'No internet service']",No,43.9


## 4. Bivariate Analysis — Key Behavioral Drivers of Churn

We investigate the operational, financial, and product features most strongly associated with customer defection.

In [7]:
key_drivers = ['Contract', 'InternetService', 'PaymentMethod', 'SeniorCitizen']

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for idx, col in enumerate(key_drivers):
    rate_df = df.groupby(col)['Churn_binary'].agg(
        churn_rate=lambda x: x.mean() * 100,
        total='count'
    ).reset_index().sort_values(by='churn_rate', ascending=False)
    
    ax = axes[idx]
    sns.barplot(data=rate_df, x=col, y='churn_rate', palette='Reds_r', ax=ax, edgecolor='black', hue=col, legend=False)
    ax.set_title(f"Churn Rate by {col}", weight='bold', fontsize=12)
    ax.set_ylabel("Churn Rate (%)")
    ax.set_ylim(0, max(rate_df['churn_rate']) * 1.25)
    if col == 'PaymentMethod':
        ax.tick_params(axis='x', rotation=20)
    for p, tot in zip(ax.patches, rate_df['total']):
        h = p.get_height()
        ax.annotate(f"{h:.1f}%\n(N={tot:,})", (p.get_x() + p.get_width()/2., h + 1),
                    ha='center', va='bottom', fontsize=9, weight='bold')

plt.tight_layout()
plt.show()

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_36256\3551276311.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

sns.kdeplot(data=df, x='tenure', hue='Churn', common_norm=False, fill=True, ax=ax1, palette={'No': '#4C72B0', 'Yes': '#DD8452'})
ax1.set_title("Tenure Distribution by Churn Status", weight='bold')
ax1.set_xlabel("Tenure (Months)")

sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', common_norm=False, fill=True, ax=ax2, palette={'No': '#4C72B0', 'Yes': '#DD8452'})
ax2.set_title("Monthly Charges by Churn Status", weight='bold')
ax2.set_xlabel("Monthly Charges ($)")

plt.tight_layout()
plt.show()

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_36256\761794801.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Association Analysis: Cramér's V & Numeric Correlations

> **Statistical Integrity Rule:** Pearson correlation requires continuous interval/ratio data with linear relationships. It is mathematically invalid to compute Pearson correlation on nominal categorical variables. For nominal-to-nominal/target association, we use **Cramér's V with Bergsma bias correction**.

In [9]:
cramers_results = {}
for col in cat_cols:
    tbl = pd.crosstab(df[col], df['Churn'])
    cramers_results[col] = calculate_cramers_v(tbl)

df_cv = pd.DataFrame(list(cramers_results.items()), columns=['Feature', 'Cramers_V'])
df_cv = df_cv.sort_values(by='Cramers_V', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 5.5))
norm = plt.Normalize(df_cv['Cramers_V'].min(), df_cv['Cramers_V'].max())
colors = plt.cm.viridis(norm(df_cv['Cramers_V']))

bars = plt.barh(df_cv['Feature'][::-1], df_cv['Cramers_V'][::-1], color=colors[::-1], edgecolor='black', linewidth=0.5)
plt.title("Categorical Association with Churn (Cramér's V)", weight='bold', fontsize=12)
plt.xlabel("Cramér's V (Bias-Corrected)")
plt.xlim(0, max(df_cv['Cramers_V']) * 1.15)
for bar in bars:
    w = bar.get_width()
    plt.annotate(f" {w:.3f}", (w, bar.get_y() + bar.get_height()/2), ha='left', va='center', weight='bold', fontsize=9)
plt.tight_layout()
plt.show()

df_cv

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_36256\4205746701.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,Feature,Cramers_V
0,Contract,0.409798
1,OnlineSecurity,0.347016
2,TechSupport,0.342526
3,InternetService,0.322037
4,PaymentMethod,0.302677
5,OnlineBackup,0.291850
6,DeviceProtection,0.281095
7,StreamingMovies,0.230351
8,StreamingTV,0.229902
9,PaperlessBilling,0.191141


In [10]:
corr_num = df[['tenure', 'MonthlyCharges', 'TotalCharges_num', 'Churn_binary']].corr(method='spearman')

plt.figure(figsize=(7, 5))
sns.heatmap(corr_num, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1)
plt.title("Spearman Rank Correlation (Numeric Features + Churn)", weight='bold')
plt.tight_layout()
plt.show()

C:\Users\Abhishek\AppData\Local\Temp\ipykernel_36256\4258475342.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Outlier Detection & Domain Classification

We detect numeric values outside $[Q_1 - 1.5 \times \text{IQR}, Q_3 + 1.5 \times \text{IQR}]$ and classify each observation instead of blindly deleting them.

In [11]:
outlier_report = []
for c in num_cols:
    s = df[c].dropna()
    q25, q75 = s.quantile(0.25), s.quantile(0.75)
    iqr = q75 - q25
    low, high = q25 - 1.5 * iqr, q75 + 1.5 * iqr
    outliers = s[(s < low) | (s > high)]
    outlier_report.append({
        'Feature': c,
        'Lower Bound': round(low, 2),
        'Upper Bound': round(high, 2),
        'Outlier Count': len(outliers),
        'Outlier %': f"{(len(outliers) / len(s)) * 100:.2f}%",
        'Domain Classification': 'Valid observation (within authentic business constraints)'
    })

pd.DataFrame(outlier_report)

,Feature,Lower Bound,Upper Bound,Outlier Count,Outlier %,Domain Classification
0,tenure,-60.00,124.00,0,0.00%,Valid observation (within authentic business c...
1,MonthlyCharges,-46.02,171.38,0,0.00%,Valid observation (within authentic business c...
2,TotalCharges_num,-4688.48,8884.67,0,0.00%,Valid observation (within authentic business c...


## 7. Investigation of 11 Blank TotalCharges Values

In Phase 2, data validation flagged 11 records where `TotalCharges` was whitespace `" "`. We inspect these records here.

In [12]:
blank_mask = df_raw['TotalCharges'].str.strip() == ''
blank_rows = df_raw[blank_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'Churn']]
print(f"Number of blank TotalCharges rows: {len(blank_rows)}")
print(f"Unique tenure values for these rows: {blank_rows['tenure'].unique()}")
blank_rows

Number of blank TotalCharges rows: 11
Unique tenure values for these rows: [0]


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


### Root Cause & Cleaning Decision
- **Root Cause:** All 11 records have `tenure = 0`. These are brand-new customers who registered during the current billing period and have not yet been billed for a full cycle.
- **Decision for Phase 4:** Impute `0.0` for `TotalCharges` on `tenure = 0` rows, and cast `TotalCharges` to `float64`.
- **Leakage Integrity:** This is a deterministic domain correction based on physical business logic, not a learned sample statistic, so it carries zero risk of data leakage.

---

## 8. Summary of Findings & Actionable Directives

### Key Insights
1. **Contract Type is the Dominant Factor ($V = 0.410$):** Month-to-month contracts churn at **42.7%**, compared to **11.3%** for 1-year and **2.8%** for 2-year commitments.
2. **Early Tenure Vulnerability:** Churn is heavily front-loaded in the first 12 months. Once a customer survives past month 24, churn drops substantially.
3. **Product Fit & Tech Support ($V = 0.342$):** Customers without TechSupport or OnlineSecurity churn at ~41%, whereas those with support churn at ~15%.
4. **Fiber Optic Paradox:** Fiber optic customers churn at **41.9%** despite having the highest ARPU, indicating either service delivery friction or intense competitor switching.
5. **Payment Method ($V = 0.303$):** Electronic check users churn at **45.3%** vs ~15-16% for automated credit card or bank transfer users.

### Directives for Phase 4 (Data Cleaning)
- Cast `TotalCharges` to float64, filling the 11 blank records with `0.0`.
- Drop `customerID` (unique identifier with no generalizable signal).
- Map target `Churn` to binary integer (`Yes` $\to 1$, `No` $\to 0$).

### Directives for Phase 5 (Feature Engineering)
- `tenure_cohort`: Bin tenure into customer lifecycle stages (`0-12m`, `13-24m`, `25-48m`, `49-60m`, `>60m`).
- `is_auto_pay`: Binary flag for automatic bank or credit card payment.
- `total_services`: Aggregate count of subscribed add-on services (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`).
- `charges_per_tenure`: Ratio of `TotalCharges` to `(tenure + 1)`.
- `has_fiber_optic`: Flag for high-risk fiber optic service.